In [10]:
import os
import shutil
import torch
from torchvision import datasets
from sklearn.model_selection import train_test_split

# ✅ 기존 데이터셋 폴더 (변경 ❌)
original_dataset_dir = "C:/Users/user/OneDrive/Desktop/Camter_IAI/tent-classification/data/camter_tent_data2"

# ✅ 새로운 데이터셋 폴더 (새롭게 생성됨 ✅)
new_dataset_dir = "C:/Users/user/OneDrive/Desktop/Camter_IAI/tent-classification/data/last_camter_tent_dataset"
train_dir = os.path.join(new_dataset_dir, "train")
val_dir = os.path.join(new_dataset_dir, "val")
test_dir = os.path.join(new_dataset_dir, "test")

# ✅ 새로운 폴더 생성 (train, val, test)
for dir_path in [train_dir, val_dir, test_dir]:
    os.makedirs(dir_path, exist_ok=True)

# ✅ 기존 데이터셋 폴더가 존재하는지 확인 (오류 방지)
if not os.path.exists(original_dataset_dir):
    raise FileNotFoundError(f"❌ 오류: {original_dataset_dir} 폴더를 찾을 수 없습니다. 경로를 확인하세요.")

# ✅ 기존 데이터셋 로드 (이미 브랜드별 폴더 구조가 있어야 함)
if len(os.listdir(original_dataset_dir)) == 0:
    raise FileNotFoundError(f"❌ 오류: {original_dataset_dir} 폴더 내부가 비어 있습니다.")

# ✅ 올바른 폴더 구조인지 확인
brand_folders = [f for f in os.listdir(original_dataset_dir) if os.path.isdir(os.path.join(original_dataset_dir, f))]
if len(brand_folders) == 0:
    raise FileNotFoundError("❌ 오류: 브랜드별 폴더가 존재하지 않습니다. 올바른 폴더 구조를 확인하세요.")

# ✅ 브랜드별로 데이터를 정리하여 새로운 폴더로 복사
brand_images = {}
for brand in brand_folders:
    brand_path = os.path.join(original_dataset_dir, brand)
    images = [os.path.join(brand_path, f) for f in os.listdir(brand_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    if len(images) == 0:
        print(f"⚠ 브랜드 '{brand}' 폴더에 이미지가 없습니다. 건너뜁니다.")
        continue

    brand_images[brand] = images

# ✅ 데이터셋을 새로운 폴더로 정리
for brand, images in brand_images.items():
    brand_train_dir = os.path.join(train_dir, brand)
    brand_val_dir = os.path.join(val_dir, brand)
    brand_test_dir = os.path.join(test_dir, brand)

    os.makedirs(brand_train_dir, exist_ok=True)
    os.makedirs(brand_val_dir, exist_ok=True)
    os.makedirs(brand_test_dir, exist_ok=True)

    if len(images) < 5:
        print(f"⚠ 브랜드 {brand}에 이미지가 너무 적습니다. 최소 5장 이상 필요합니다.")
        continue

    test_images = images[:4]  # 브랜드별 4장 테스트 데이터
    train_val_images = images[4:]  # 나머지 데이터 (훈련 + 검증)

    # 훈련/검증 데이터 나누기 (8:2 비율)
    train_images, val_images = train_test_split(train_val_images, test_size=0.2, random_state=42)

    # ✅ 기존 데이터셋을 유지하면서 새 폴더에 복사 (변경 ❌)
    for img_path in test_images:
        shutil.copy(img_path, os.path.join(brand_test_dir, os.path.basename(img_path)))
    for img_path in train_images:
        shutil.copy(img_path, os.path.join(brand_train_dir, os.path.basename(img_path)))
    for img_path in val_images:
        shutil.copy(img_path, os.path.join(brand_val_dir, os.path.basename(img_path)))

print(f"✅ 데이터셋이 성공적으로 새로운 폴더에 저장되었습니다: {new_dataset_dir}")


⚠ 브랜드 'test' 폴더에 이미지가 없습니다. 건너뜁니다.
⚠ 브랜드 'train' 폴더에 이미지가 없습니다. 건너뜁니다.
⚠ 브랜드 'val' 폴더에 이미지가 없습니다. 건너뜁니다.
✅ 데이터셋이 성공적으로 새로운 폴더에 저장되었습니다: C:/Users/user/OneDrive/Desktop/Camter_IAI/tent-classification/data/last_camter_tent_dataset


In [13]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# ✅ GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 사용 중인 디바이스: {device}")

# ✅ 데이터셋 경로 설정
dataset_dir = "C:/Users/user/OneDrive/Desktop/Camter_IAI/tent-classification/data/last_camter_tent_dataset"
train_dir = os.path.join(dataset_dir, "train")
val_dir = os.path.join(dataset_dir, "val")

# ✅ 데이터 전처리 (EfficientNet-B0 기본 설정)
transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ✅ 데이터셋 로드
train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
val_dataset = datasets.ImageFolder(root=val_dir, transform=transform)

# ✅ 데이터 로더 설정
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ✅ EfficientNet-B0 모델 로드
from torchvision.models import EfficientNet_B0_Weights

model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
num_classes = len(train_dataset.classes)
model.classifier = nn.Sequential(
    nn.Linear(model.classifier[1].in_features, num_classes)
)
model = model.to(device)

# ✅ 손실 함수 및 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ✅ 모델 저장 경로 설정
model_dir = "C:/Users/user/OneDrive/Desktop/Camter_IAI/tent-classification/models"
os.makedirs(model_dir, exist_ok=True)
best_model_path = os.path.join(model_dir, "best_efficientnet_tent_model.pth")

# ✅ 최고 검증 정확도 및 손실값 추적
best_val_accuracy = 0.0
best_val_loss = float("inf")

# ✅ 학습 및 검증 진행
EPOCHS = 300
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct_train += (predicted == labels).sum().item()
        total_train += labels.size(0)

    train_accuracy = correct_train / total_train
    train_losses.append(running_loss / len(train_loader))
    train_accuracies.append(train_accuracy)

    # ✅ 검증 단계
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct_val += (predicted == labels).sum().item()
            total_val += labels.size(0)

    val_accuracy = correct_val / total_val
    val_losses.append(val_loss / len(val_loader))
    val_accuracies.append(val_accuracy)

    # ✅ 출력 형식을 한 줄로 정리
    print(f"📢 Epoch [{epoch + 1}/{EPOCHS}] | "
        f"학습 손실: {running_loss/len(train_loader):.4f} | "
        f"학습 정확도: {train_accuracy:.4f} | "
        f"검증 손실: {val_loss/len(val_loader):.4f} | "
        f"검증 정확도: {val_accuracy:.4f}")

    # ✅ 최고 검증 정확도 모델 저장
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_val_loss = val_loss / len(val_loader)
        torch.save(model.state_dict(), best_model_path)
        print(f"🎯 새로운 최고 정확도 모델 저장됨! (정확도: {best_val_accuracy:.4f}, 손실값: {best_val_loss:.4f})")

# ✅ 학습 완료 후 최고 검증 정확도 및 손실값 출력
print("\n🚀 학습 완료!")
print(f"✅ 최고 검증 정확도: {best_val_accuracy:.4f}")
print(f"✅ 최고 검증 손실값: {best_val_loss:.4f}")
print(f"📂 최고 정확도 모델 저장 경로: {best_model_path}")


🚀 사용 중인 디바이스: cuda
📢 Epoch [1/300] | 학습 손실: 1.5279 | 학습 정확도: 0.3918 | 검증 손실: 1.0905 | 검증 정확도: 0.6170
🎯 새로운 최고 정확도 모델 저장됨! (정확도: 0.6170, 손실값: 1.0905)
📢 Epoch [2/300] | 학습 손실: 0.9180 | 학습 정확도: 0.6822 | 검증 손실: 1.1135 | 검증 정확도: 0.6170
📢 Epoch [3/300] | 학습 손실: 0.7253 | 학습 정확도: 0.7534 | 검증 손실: 1.1437 | 검증 정확도: 0.6489
🎯 새로운 최고 정확도 모델 저장됨! (정확도: 0.6489, 손실값: 1.1437)
📢 Epoch [4/300] | 학습 손실: 0.5467 | 학습 정확도: 0.8301 | 검증 손실: 1.1615 | 검증 정확도: 0.6596
🎯 새로운 최고 정확도 모델 저장됨! (정확도: 0.6596, 손실값: 1.1615)
📢 Epoch [5/300] | 학습 손실: 0.6336 | 학습 정확도: 0.7781 | 검증 손실: 0.8645 | 검증 정확도: 0.7340
🎯 새로운 최고 정확도 모델 저장됨! (정확도: 0.7340, 손실값: 0.8645)
📢 Epoch [6/300] | 학습 손실: 0.5574 | 학습 정확도: 0.8164 | 검증 손실: 0.9780 | 검증 정확도: 0.7021
📢 Epoch [7/300] | 학습 손실: 0.4505 | 학습 정확도: 0.8356 | 검증 손실: 1.1900 | 검증 정확도: 0.6489
📢 Epoch [8/300] | 학습 손실: 0.4806 | 학습 정확도: 0.8329 | 검증 손실: 1.2150 | 검증 정확도: 0.7128
📢 Epoch [9/300] | 학습 손실: 0.4282 | 학습 정확도: 0.8548 | 검증 손실: 1.0036 | 검증 정확도: 0.7340
📢 Epoch [10/300] | 학습 손실: 0.4962 | 학습 정확도: 0.8192 |

In [16]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import numpy as np

# ✅ GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 사용 중인 디바이스: {device}")

# ✅ 데이터셋 경로 설정
dataset_dir = "C:/Users/user/OneDrive/Desktop/Camter_IAI/tent-classification/data/last_camter_tent_dataset"
test_dir = os.path.join(dataset_dir, "test")

# ✅ 모델이 예측할 수 있는 브랜드 목록 (수동 지정)
class_names = [
    "0 헬리녹스 알파인돔", 
    "1 헬리녹스 노나돔", 
    "2 헬리녹스 브이타프",
    "3 스노우피크 도크돔",
    "9 힐레베르그 알락",
    "10 헬스포츠 김레패밀리"
]
print(f"📂 모델이 예측할 수 있는 브랜드 목록: {class_names}")

# ✅ 데이터 전처리 (훈련 데이터와 동일)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ✅ 테스트 데이터셋 로드
test_dataset = datasets.ImageFolder(root=test_dir, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# ✅ 저장된 최적 가중치 로드
model_path = "C:/Users/user/OneDrive/Desktop/Camter_IAI/tent-classification/models/best_efficientnet_tent_model.pth"

# ✅ EfficientNet-B0 모델 로드
from torchvision.models import EfficientNet_B0_Weights

model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
num_classes = len(class_names)
model.classifier = nn.Sequential(
    nn.Linear(model.classifier[1].in_features, num_classes)
)
model.load_state_dict(torch.load(model_path))  # 가중치 로드
model = model.to(device)
model.eval()  # 평가 모드로 전환

# ✅ 테스트 데이터셋 평가
correct = 0
total = 0
brand_correct = {brand: 0 for brand in class_names}  # 브랜드별 정답 개수
brand_total = {brand: 0 for brand in class_names}  # 브랜드별 총 개수

print("\n📢 [테스트 결과]")
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        actual_brand = class_names[labels.item()]
        predicted_brand = class_names[predicted.item()]
        
        # ✅ 총 정확도 계산
        total += 1
        if predicted.item() == labels.item():
            correct += 1
            brand_correct[actual_brand] += 1
        
        brand_total[actual_brand] += 1
        
        print(f"🔍 실제: {actual_brand} | 예측: {predicted_brand} | {'✅ 정답' if predicted_brand == actual_brand else '❌ 오답'}")

# ✅ 브랜드별 정확도 출력
print("\n📊 [브랜드별 테스트 정확도]")
for brand in class_names:
    if brand_total[brand] > 0:
        accuracy = (brand_correct[brand] / brand_total[brand]) * 100
        print(f"🏕️ {brand}: {accuracy:.2f}% ({brand_correct[brand]} / {brand_total[brand]})")

# ✅ 전체 테스트 정확도 출력
test_accuracy = (correct / total) * 100
print(f"\n🎯 전체 테스트 정확도: {test_accuracy:.2f}% ({correct} / {total})")


🚀 사용 중인 디바이스: cuda
📂 모델이 예측할 수 있는 브랜드 목록: ['0 헬리녹스 알파인돔', '1 헬리녹스 노나돔', '2 헬리녹스 브이타프', '3 스노우피크 도크돔', '9 힐레베르그 알락', '10 헬스포츠 김레패밀리']


C:\Users\user\AppData\Local\Temp\ipykernel_20264\2558078599.py:49: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))  # 가중치 로드



📢 [테스트 결과]
🔍 실제: 0 헬리녹스 알파인돔 | 예측: 0 헬리녹스 알파인돔 | ✅ 정답
🔍 실제: 0 헬리녹스 알파인돔 | 예측: 1 헬리녹스 노나돔 | ❌ 오답
🔍 실제: 0 헬리녹스 알파인돔 | 예측: 0 헬리녹스 알파인돔 | ✅ 정답
🔍 실제: 0 헬리녹스 알파인돔 | 예측: 0 헬리녹스 알파인돔 | ✅ 정답
🔍 실제: 1 헬리녹스 노나돔 | 예측: 1 헬리녹스 노나돔 | ✅ 정답
🔍 실제: 1 헬리녹스 노나돔 | 예측: 1 헬리녹스 노나돔 | ✅ 정답
🔍 실제: 1 헬리녹스 노나돔 | 예측: 1 헬리녹스 노나돔 | ✅ 정답
🔍 실제: 1 헬리녹스 노나돔 | 예측: 0 헬리녹스 알파인돔 | ❌ 오답
🔍 실제: 2 헬리녹스 브이타프 | 예측: 2 헬리녹스 브이타프 | ✅ 정답
🔍 실제: 2 헬리녹스 브이타프 | 예측: 2 헬리녹스 브이타프 | ✅ 정답
🔍 실제: 2 헬리녹스 브이타프 | 예측: 2 헬리녹스 브이타프 | ✅ 정답
🔍 실제: 2 헬리녹스 브이타프 | 예측: 2 헬리녹스 브이타프 | ✅ 정답
🔍 실제: 3 스노우피크 도크돔 | 예측: 3 스노우피크 도크돔 | ✅ 정답
🔍 실제: 3 스노우피크 도크돔 | 예측: 3 스노우피크 도크돔 | ✅ 정답
🔍 실제: 3 스노우피크 도크돔 | 예측: 9 힐레베르그 알락 | ❌ 오답
🔍 실제: 3 스노우피크 도크돔 | 예측: 3 스노우피크 도크돔 | ✅ 정답
🔍 실제: 9 힐레베르그 알락 | 예측: 2 헬리녹스 브이타프 | ❌ 오답
🔍 실제: 9 힐레베르그 알락 | 예측: 9 힐레베르그 알락 | ✅ 정답
🔍 실제: 9 힐레베르그 알락 | 예측: 9 힐레베르그 알락 | ✅ 정답
🔍 실제: 9 힐레베르그 알락 | 예측: 9 힐레베르그 알락 | ✅ 정답
🔍 실제: 10 헬스포츠 김레패밀리 | 예측: 10 헬스포츠 김레패밀리 | ✅ 정답
🔍 실제: 10 헬스포츠 김레패밀리 | 예측: 10 헬스포츠 김레패밀리 | ✅ 정답
🔍 실제: 10 헬스포츠 김레패밀리 | 예측: 10 헬스포츠 김레패밀리 | ✅ 정답
🔍 실